# Diagnóstico multimodal IU X-Ray — V6.1 Reporting, GitHub Release y Demo

Este notebook corresponde a la **etapa posterior al entrenamiento**. No vuelve a entrenar el modelo. Toma el archivo:

```text
outputs_multimodal_iuxray_v6_1.zip
```

y genera:

1. Figuras listas para presentación del pipeline.
2. Tablas y métricas finales, incluyendo métricas adicionales.
3. Comparación de modelos y validación cruzada.
4. Análisis de errores y evidencia visual.
5. Carpeta lista para subir a GitHub con código fuente, README, instrucciones del dataset y pasos de ejecución.
6. Guía de demo para la exposición.

**Resultado esperado:**

```text
/content/reporting_release_v6_1/
├── figures/
├── tables/
├── docs/
├── github_repo/
├── demo/
├── presentation_assets_v6_1.zip
└── multimodal_iuxray_v6_1_github_repo_ready.zip
```

## 0. Configuración general

Sube `outputs_multimodal_iuxray_v6_1.zip` al entorno de Colab o colócalo en `/content/` antes de ejecutar.

In [ ]:
# =============================
# 0) Configuración
# =============================
from pathlib import Path
import os, re, json, zipfile, shutil, textwrap, math, sys, subprocess

VERSION = "v6_1"
ZIP_NAME = f"outputs_multimodal_iuxray_{VERSION}.zip"

# Modelo y umbral que se reportarán como resultado operativo principal.
# En V6.1 el umbral principal se definió para alta sensibilidad.
PRIMARY_MODEL = "image_calibrated"
SECONDARY_MODELS = ["fusion_weighted", "fusion_stack", "text_calibrated"]
PRIMARY_THRESHOLD_TYPE = "min_fpr_recall_0.90"

# Directorios de trabajo.
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPORT_DIR = BASE_DIR / f"reporting_release_{VERSION}"
EXTRACT_DIR = REPORT_DIR / "extracted_outputs"
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
DOCS_DIR = REPORT_DIR / "docs"
DEMO_DIR = REPORT_DIR / "demo"
GITHUB_DIR = REPORT_DIR / "github_repo"

for d in [REPORT_DIR, EXTRACT_DIR, FIG_DIR, TABLE_DIR, DOCS_DIR, DEMO_DIR, GITHUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("VERSION:", VERSION)
print("REPORT_DIR:", REPORT_DIR)

In [ ]:
# =============================
# 1) Instalación/importaciones ligeras
# =============================
# No se instala nada pesado por defecto. Si luego quieres usar demo con Gradio,
# activa INSTALL_GRADIO=True en la sección de demo.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

print("numpy", np.__version__)
print("pandas", pd.__version__)

## 1. Cargar ZIP de outputs V6.1

El notebook intentará encontrar el ZIP en rutas comunes. Si no lo encuentra en Colab, abrirá el diálogo para subirlo manualmente.

In [ ]:
# =============================
# 2) Localizar y extraer ZIP
# =============================
def find_zip(zip_name: str) -> Path:
    candidates = [
        Path("/content") / zip_name,
        Path.cwd() / zip_name,
        Path("/mnt/data") / zip_name,
        REPORT_DIR / zip_name,
    ]
    for p in candidates:
        if p.exists():
            return p

    # Fallback para Colab: subir manualmente.
    try:
        from google.colab import files
        print(f"No encontré {zip_name}. Súbelo ahora:")
        uploaded = files.upload()
        if zip_name not in uploaded:
            raise FileNotFoundError(f"Subiste {list(uploaded.keys())}, pero esperaba {zip_name}")
        return Path("/content") / zip_name
    except Exception as e:
        raise FileNotFoundError(
            f"No encontré {zip_name}. Colócalo en /content/ o en el directorio actual. Detalle: {e}"
        )

ZIP_PATH = find_zip(ZIP_NAME)
print("ZIP encontrado:", ZIP_PATH)

# Extraer limpio.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

all_files = sorted([p.relative_to(EXTRACT_DIR).as_posix() for p in EXTRACT_DIR.rglob("*") if p.is_file()])
print(f"Archivos extraídos: {len(all_files)}")
for f in all_files[:20]:
    print(" -", f)

## 2. Cargar tablas principales

In [ ]:
# =============================
# 3) Helpers de carga
# =============================
def first_existing(patterns, required=True):
    if isinstance(patterns, str):
        patterns = [patterns]
    for pat in patterns:
        matches = sorted(EXTRACT_DIR.glob(pat))
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(f"No encontré patrones: {patterns}")
    return None

def read_csv_pattern(patterns, required=True):
    p = first_existing(patterns, required=required)
    if p is None:
        return None
    df = pd.read_csv(p)
    print(f"{p.relative_to(EXTRACT_DIR)} -> {df.shape}")
    return df

def read_json_pattern(patterns, required=False):
    p = first_existing(patterns, required=required)
    if p is None:
        return None
    with open(p, "r", encoding="utf-8") as f:
        obj = json.load(f)
    print(f"{p.relative_to(EXTRACT_DIR)} -> JSON")
    return obj

summary_main = read_csv_pattern([f"main/summary_main_{VERSION}.csv", "main/summary_main_*.csv"])
thresholds_main = read_csv_pattern([f"main/thresholds_main_{VERSION}.csv", "main/thresholds_main_*.csv"])
cv_primary = read_csv_pattern([f"cv_primary_aggregate_{VERSION}.csv", "cv_primary_aggregate_*.csv"], required=False)
cv_summary = read_csv_pattern([f"cv_summary_{VERSION}.csv", "cv_summary_*.csv"], required=False)
bootstrap_ci = read_csv_pattern([f"main/bootstrap_grouped_uid_ci_{VERSION}.csv", "main/bootstrap_grouped_uid_ci_*.csv"], required=False)
mcnemar = read_csv_pattern([f"main/mcnemar_primary_threshold_{VERSION}.csv", "main/mcnemar_primary_threshold_*.csv"], required=False)
error_counts = read_csv_pattern([f"main/error_counts_primary_threshold_{VERSION}.csv", "main/error_counts_primary_threshold_*.csv"], required=False)
error_analysis = read_csv_pattern([f"main/error_analysis_test_{VERSION}.csv", "main/error_analysis_test_*.csv"], required=False)
hardest_errors = read_csv_pattern([f"main/hardest_error_cases_{VERSION}.csv", "main/hardest_error_cases_*.csv"], required=False)
pred_test_uid = read_csv_pattern([f"main/predictions_test_uid_main_{VERSION}.csv", "main/predictions_test_uid_main_*.csv"], required=False)
training_history = read_csv_pattern([f"main/training_history_main_{VERSION}.csv", "main/training_history_main_*.csv"], required=False)
fusion_selected = read_json_pattern([f"main/fusion_weight_selected_main_{VERSION}.json", "main/fusion_weight_selected_main_*.json"], required=False)

dataset_summary = read_csv_pattern([f"dataset_summary_{VERSION}.csv", "dataset_summary_*.csv"])
split_summary = read_csv_pattern([f"split_summary_{VERSION}.csv", "split_summary_*.csv"])
audit_summary = read_csv_pattern([f"label_audit_summary_{VERSION}.csv", "label_audit_summary_*.csv"])
category_distribution = read_csv_pattern([f"category_distribution_before_filter_{VERSION}.csv", "category_distribution_before_filter_*.csv"], required=False)
benign_review = read_csv_pattern([f"benign_nodular_review_candidates_{VERSION}.csv", "benign_nodular_review_candidates_*.csv"], required=False)
config = read_json_pattern([f"config_{VERSION}.json", "config_*.json"], required=False)
manifest = read_json_pattern([f"manifest_{VERSION}.json", "manifest_*.json"], required=False)

## 3. Métricas finales y métricas adicionales

Además de ROC-AUC, PR-AUC, sensibilidad, especificidad, FPR y Brier, se calculan métricas adicionales útiles para reporte:

- Accuracy
- Precision / PPV
- NPV
- F1-score
- Balanced accuracy
- MCC
- Youden J
- FNR
- LR+ y LR-
- FDR y FOR

In [ ]:
# =============================
# 4) Métricas adicionales
# =============================
def safe_div(a, b):
    return np.nan if b == 0 else a / b

def add_additional_metrics(df):
    df = df.copy()
    if not set(["tn", "fp", "fn", "tp"]).issubset(df.columns):
        return df
    rows = []
    for _, r in df.iterrows():
        tn, fp, fn, tp = int(r["tn"]), int(r["fp"]), int(r["fn"]), int(r["tp"])
        sens = safe_div(tp, tp + fn)
        spec = safe_div(tn, tn + fp)
        fpr = safe_div(fp, fp + tn)
        fnr = safe_div(fn, fn + tp)
        ppv = safe_div(tp, tp + fp)
        npv = safe_div(tn, tn + fn)
        acc = safe_div(tp + tn, tp + tn + fp + fn)
        f1 = safe_div(2 * tp, 2 * tp + fp + fn)
        bal_acc = np.nanmean([sens, spec])
        denom = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
        mcc = safe_div((tp*tn - fp*fn), denom)
        youden = sens + spec - 1 if not (np.isnan(sens) or np.isnan(spec)) else np.nan
        lr_pos = safe_div(sens, fpr)
        lr_neg = safe_div(fnr, spec)
        fdr = safe_div(fp, fp + tp)
        forate = safe_div(fn, fn + tn)
        d = r.to_dict()
        d.update({
            "ppv_precision": ppv,
            "npv": npv,
            "balanced_accuracy": bal_acc,
            "mcc": mcc,
            "youden_j": youden,
            "lr_positive": lr_pos,
            "lr_negative": lr_neg,
            "false_discovery_rate": fdr,
            "false_omission_rate": forate,
        })
        rows.append(d)
    return pd.DataFrame(rows)

# Tabla principal UID con umbral operativo.
main_uid_primary = summary_main[
    (summary_main["split"].eq("test_uid")) &
    (summary_main["threshold_type"].eq(PRIMARY_THRESHOLD_TYPE))
].copy()

model_order = ["image_calibrated", "fusion_weighted", "fusion_stack", "text_calibrated", "image_raw"]
main_uid_primary["model_order"] = main_uid_primary["model"].apply(lambda x: model_order.index(x) if x in model_order else 999)
main_uid_primary = main_uid_primary.sort_values("model_order").drop(columns=["model_order"])
main_uid_primary_additional = add_additional_metrics(main_uid_primary)

# Guardar tablas.
main_uid_primary.to_csv(TABLE_DIR / f"final_metrics_test_uid_{VERSION}.csv", index=False)
main_uid_primary_additional.to_csv(TABLE_DIR / f"additional_metrics_test_uid_{VERSION}.csv", index=False)
thresholds_main.to_csv(TABLE_DIR / f"thresholds_main_{VERSION}.csv", index=False)
if cv_primary is not None:
    cv_primary.to_csv(TABLE_DIR / f"cv_primary_aggregate_{VERSION}.csv", index=False)
if bootstrap_ci is not None:
    bootstrap_ci.to_csv(TABLE_DIR / f"bootstrap_grouped_uid_ci_{VERSION}.csv", index=False)
if mcnemar is not None:
    mcnemar.to_csv(TABLE_DIR / f"mcnemar_primary_threshold_{VERSION}.csv", index=False)

print("Métricas UID primarias:")
display(main_uid_primary[[
    "model", "roc_auc", "pr_auc", "brier", "accuracy", "precision", "recall_sensitivity",
    "specificity", "false_positive_rate", "false_negative_rate", "f1", "tn", "fp", "fn", "tp"
]])

print("Métricas adicionales:")
display(main_uid_primary_additional[[
    "model", "npv", "balanced_accuracy", "mcc", "youden_j", "lr_positive", "lr_negative",
    "false_discovery_rate", "false_omission_rate"
]])

## 4. Figuras para presentación del pipeline

Estas figuras cubren lo que normalmente se solicita en la exposición:

- Data input
- Preprocessing
- Model
- Training
- Evaluation
- Final output

In [ ]:
# =============================
# 5) Figuras: pipeline y arquitectura
# =============================
def save_current_fig(name, dpi=220):
    path_png = FIG_DIR / f"{name}.png"
    path_svg = FIG_DIR / f"{name}.svg"
    plt.tight_layout()
    plt.savefig(path_png, dpi=dpi, bbox_inches="tight")
    plt.savefig(path_svg, bbox_inches="tight")
    plt.show()
    print("Guardado:", path_png)
    return path_png


def draw_pipeline_flow():
    steps = [
        ("Data input", "IU X-Ray: imagen frontal\n+ indication clínica\n+ metadata UID"),
        ("Preprocessing", "Normalización, split por UID,\nauditoría de etiquetas, exclusiones"),
        ("Model", "Rama visual CNN\n+ rama texto segura\n+ calibración"),
        ("Training", "Train/val/calibration/threshold/test\nCV por UID"),
        ("Evaluation", "ROC-AUC, PR-AUC, Brier,\nRecall, FPR, bootstrap, McNemar"),
        ("Final output", "Modelo principal: image_calibrated\nTablas, curvas, demo, ZIP GitHub"),
    ]
    fig, ax = plt.subplots(figsize=(16, 4.8))
    ax.axis("off")
    x_positions = np.linspace(0.02, 0.83, len(steps))
    y = 0.45
    w, h = 0.145, 0.36
    for i, (title, body) in enumerate(steps):
        x = x_positions[i]
        box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.025", linewidth=1.5, facecolor="white")
        ax.add_patch(box)
        ax.text(x + w/2, y + h*0.72, title, ha="center", va="center", fontsize=12, fontweight="bold")
        ax.text(x + w/2, y + h*0.38, body, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            start = (x + w + 0.005, y + h/2)
            end = (x_positions[i+1] - 0.005, y + h/2)
            arrow = FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=18, linewidth=1.5)
            ax.add_patch(arrow)
    ax.set_title("Pipeline completo del proyecto", fontsize=16, fontweight="bold", pad=14)
    save_current_fig("pipeline_project_flow")


def draw_model_architecture():
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.axis("off")

    boxes = [
        (0.03, 0.63, 0.19, 0.18, "Imagen CXR frontal", "PNG/DICOM normalizado\nIMAGE_SIZE=320"),
        (0.03, 0.20, 0.19, 0.18, "Indication", "Texto clínico mínimo\nsin findings/impression"),
        (0.31, 0.63, 0.19, 0.18, "Rama visual", "CNN preentrenada\nclasificación binaria"),
        (0.31, 0.20, 0.19, 0.18, "Rama textual", "TF-IDF / modelo lineal\nseñal auxiliar"),
        (0.59, 0.63, 0.18, 0.18, "Calibración", "Platt / calibración\nprobabilidad interpretable"),
        (0.59, 0.20, 0.18, 0.18, "Fusión", "weighted fusion / stacking\nvalidado en split separado"),
        (0.84, 0.43, 0.13, 0.20, "Salida", "P(anormalidad)\n+ umbral operativo"),
    ]
    for x, y, w, h, title, body in boxes:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.025", linewidth=1.5, facecolor="white"))
        ax.text(x+w/2, y+h*0.68, title, ha="center", va="center", fontsize=12, fontweight="bold")
        ax.text(x+w/2, y+h*0.35, body, ha="center", va="center", fontsize=9)

    arrows = [
        ((0.22,0.72),(0.31,0.72)), ((0.22,0.29),(0.31,0.29)),
        ((0.50,0.72),(0.59,0.72)), ((0.50,0.29),(0.59,0.29)),
        ((0.77,0.72),(0.84,0.55)), ((0.77,0.29),(0.84,0.51)),
    ]
    for start, end in arrows:
        ax.add_patch(FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=18, linewidth=1.5))

    ax.text(0.50, 0.04, "Resultado V6.1: el selector conservador asignó peso 1.0 a imagen y 0.0 a texto; por eso el modelo principal recomendado es image_calibrated.",
            ha="center", fontsize=10)
    ax.set_title("Arquitectura evaluada", fontsize=16, fontweight="bold", pad=12)
    save_current_fig("model_architecture_v6_1")

draw_pipeline_flow()
draw_model_architecture()

## 5. Figuras de datos y auditoría

In [ ]:
# =============================
# 6) Figuras: datos, splits, auditoría
# =============================
def plot_category_distribution():
    if category_distribution is None:
        print("No hay category_distribution.")
        return
    df = category_distribution.copy().sort_values("n_rows", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(df["diagnostic_category"], df["n_rows"])
    ax.set_xlabel("Número de filas")
    ax.set_title("Distribución de categorías antes del filtro binario")
    for i, v in enumerate(df["n_rows"]):
        ax.text(v + max(df["n_rows"])*0.01, i, str(int(v)), va="center", fontsize=9)
    save_current_fig("data_category_distribution")


def plot_split_summary():
    df = split_summary.copy()
    x = np.arange(len(df))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 0.18, df["positives_rows"], width=0.36, label="Positivos")
    ax.bar(x + 0.18, df["negatives_rows"], width=0.36, label="Negativos")
    ax.set_xticks(x)
    ax.set_xticklabels(df["split"], rotation=25, ha="right")
    ax.set_ylabel("Filas")
    ax.set_title("Distribución de clases por split")
    ax.legend()
    save_current_fig("split_class_distribution")

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(df["split"], df["row_prevalence"], marker="o", label="Prevalencia por fila")
    ax.plot(df["split"], df["uid_prevalence"], marker="o", label="Prevalencia por UID")
    ax.set_ylim(0, max(0.5, df[["row_prevalence", "uid_prevalence"]].max().max() + 0.05))
    ax.set_ylabel("Prevalencia positiva")
    ax.set_title("Prevalencia por split")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    save_current_fig("split_prevalence")


def plot_audit_summary():
    df = audit_summary.copy()
    df["status"] = df["pass"].map({True:"PASS", False:"FAIL"})
    fig, ax = plt.subplots(figsize=(10, 5.8))
    y = np.arange(len(df))
    vals = df["value"].astype(float)
    ax.barh(y, vals)
    ax.set_yticks(y)
    ax.set_yticklabels(df["check"])
    ax.invert_yaxis()
    ax.set_xlabel("Valor observado")
    ax.set_title("Auditoría de etiquetas V6.1")
    for i, r in df.iterrows():
        ax.text(float(r["value"]) + max(vals.max(),1)*0.01, i, f"{r['status']} | {r['severity']}", va="center", fontsize=8)
    save_current_fig("label_audit_summary")

plot_category_distribution()
plot_split_summary()
plot_audit_summary()

## 6. Gráficos de métricas principales y adicionales

In [ ]:
# =============================
# 7) Figuras: métricas principales
# =============================
def plot_metric_group(df, metrics, title, filename, higher_is_better=True):
    plot_df = df[["model"] + metrics].copy()
    x = np.arange(len(plot_df["model"]))
    width = 0.8 / len(metrics)
    fig, ax = plt.subplots(figsize=(12, 5.5))
    for i, m in enumerate(metrics):
        ax.bar(x + (i - (len(metrics)-1)/2)*width, plot_df[m], width=width, label=m)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["model"], rotation=20, ha="right")
    ax.set_ylabel("Valor")
    ax.set_title(title)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="best")
    save_current_fig(filename)

plot_metric_group(
    main_uid_primary,
    ["roc_auc", "pr_auc", "brier"],
    f"Métricas de discriminación y calibración — test UID ({PRIMARY_THRESHOLD_TYPE})",
    "metrics_auc_pr_brier_test_uid"
)

plot_metric_group(
    main_uid_primary,
    ["recall_sensitivity", "specificity", "false_positive_rate", "precision", "f1"],
    f"Métricas operativas — test UID ({PRIMARY_THRESHOLD_TYPE})",
    "metrics_operational_test_uid"
)

plot_metric_group(
    main_uid_primary_additional,
    ["npv", "balanced_accuracy", "mcc", "youden_j"],
    "Métricas adicionales — test UID",
    "additional_metrics_test_uid"
)

In [ ]:
# =============================
# 8) Figura: matrices de confusión
# =============================
def plot_confusion_for_models(df, models):
    for model in models:
        row = df[df["model"].eq(model)]
        if row.empty:
            continue
        r = row.iloc[0]
        mat = np.array([[r["tn"], r["fp"]], [r["fn"], r["tp"]]], dtype=float)
        fig, ax = plt.subplots(figsize=(5.2, 4.6))
        im = ax.imshow(mat)
        ax.set_xticks([0,1]); ax.set_xticklabels(["Pred 0", "Pred 1"])
        ax.set_yticks([0,1]); ax.set_yticklabels(["Real 0", "Real 1"])
        ax.set_title(f"Matriz de confusión — {model}")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(int(mat[i,j])), ha="center", va="center", fontsize=14, fontweight="bold")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        save_current_fig(f"confusion_matrix_{model}")

plot_confusion_for_models(main_uid_primary, ["image_calibrated", "fusion_weighted", "fusion_stack", "text_calibrated"])

In [ ]:
# =============================
# 9) Figura: trade-off por sensibilidad objetivo
# =============================
def plot_threshold_tradeoff(summary_df):
    df = summary_df[summary_df["split"].eq("test_uid")].copy()
    keep_models = ["image_calibrated", "fusion_weighted", "fusion_stack", "text_calibrated"]
    df = df[df["model"].isin(keep_models)]
    if df.empty:
        print("No hay datos para threshold tradeoff.")
        return
    df = df.sort_values(["model", "target_recall"])

    for metric, ylab, fname in [
        ("false_positive_rate", "FPR", "threshold_tradeoff_fpr"),
        ("recall_sensitivity", "Sensibilidad observada", "threshold_tradeoff_recall"),
        ("specificity", "Especificidad", "threshold_tradeoff_specificity"),
        ("precision", "Precisión / PPV", "threshold_tradeoff_precision"),
    ]:
        fig, ax = plt.subplots(figsize=(9, 5))
        for m in keep_models:
            sub = df[df["model"].eq(m)]
            if not sub.empty:
                ax.plot(sub["target_recall"], sub[metric], marker="o", label=m)
        ax.set_xlabel("Recall objetivo usado para seleccionar umbral")
        ax.set_ylabel(ylab)
        ax.set_title(f"Trade-off de umbral: {ylab}")
        ax.grid(True, alpha=0.3)
        ax.legend()
        save_current_fig(fname)

plot_threshold_tradeoff(summary_main)

## 7. Curvas existentes: ROC, PR, calibración y entrenamiento

Se copian las curvas generadas en V6.1 hacia la carpeta de presentación.

In [ ]:
# =============================
# 10) Copiar figuras existentes del entrenamiento
# =============================
existing_fig_patterns = [
    "main_roc_curves_test_*.png",
    "main_precision_recall_curves_test_*.png",
    "main/calibration_curves_test_*.png",
    "main/image_loss_curve_*.png",
    "main/image_auc_pr_curve_*.png",
    "cv_fpr_by_fold_*.png",
    "main/fpr_by_target_recall_*.png",
    "main/recall_by_target_recall_*.png",
    "main/specificity_by_target_recall_*.png",
]

copied = []
for pat in existing_fig_patterns:
    for src in EXTRACT_DIR.glob(pat):
        dst = FIG_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst.name)

print("Figuras copiadas:")
for c in copied:
    print(" -", c)

## 8. Cross-validation, bootstrap, McNemar y errores

In [ ]:
# =============================
# 11) CV, bootstrap, McNemar, errores
# =============================
def plot_cv_aggregate():
    if cv_primary is None:
        print("No hay CV aggregate.")
        return
    df = cv_primary.copy()
    keep = ["image_calibrated", "fusion_weighted", "fusion_stack", "text_calibrated"]
    df = df[df["model"].isin(keep)]
    x = np.arange(len(df))
    for mean_col, std_col, ylab, fname in [
        ("roc_auc_mean", "roc_auc_std", "ROC-AUC medio", "cv_roc_auc_aggregate"),
        ("pr_auc_mean", "pr_auc_std", "PR-AUC medio", "cv_pr_auc_aggregate"),
        ("recall_mean", "recall_std", "Recall medio", "cv_recall_aggregate"),
        ("fpr_mean", "fpr_std", "FPR medio", "cv_fpr_aggregate"),
        ("brier_mean", "brier_std", "Brier medio", "cv_brier_aggregate"),
    ]:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(x, df[mean_col], yerr=df[std_col], capsize=5)
        ax.set_xticks(x)
        ax.set_xticklabels(df["model"], rotation=20, ha="right")
        ax.set_ylabel(ylab)
        ax.set_title(f"Validación cruzada: {ylab} ± SD")
        ax.grid(True, axis="y", alpha=0.3)
        save_current_fig(fname)


def plot_bootstrap_ci():
    if bootstrap_ci is None:
        print("No hay bootstrap CI.")
        return
    df = bootstrap_ci.copy()
    print("Columnas bootstrap:", df.columns.tolist())
    # Intentar detectar columnas de métrica/delta/IC.
    metric_col = next((c for c in ["metric", "delta_metric", "name"] if c in df.columns), None)
    mean_col = next((c for c in ["delta_mean", "mean", "estimate"] if c in df.columns), None)
    low_col = next((c for c in ["ci_low", "lower", "p2.5", "p2_5"] if c in df.columns), None)
    high_col = next((c for c in ["ci_high", "upper", "p97.5", "p97_5"] if c in df.columns), None)
    if not all([metric_col, mean_col, low_col, high_col]):
        display(df.head())
        print("No pude detectar columnas estándar para graficar bootstrap.")
        return
    df = df.dropna(subset=[mean_col, low_col, high_col]).copy()
    fig, ax = plt.subplots(figsize=(10, max(4, len(df)*0.45)))
    y = np.arange(len(df))
    ax.errorbar(df[mean_col], y, xerr=[df[mean_col]-df[low_col], df[high_col]-df[mean_col]], fmt="o", capsize=4)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(df[metric_col])
    ax.set_xlabel("Delta medio con IC 95%")
    ax.set_title("Bootstrap agrupado por UID")
    ax.grid(True, axis="x", alpha=0.3)
    save_current_fig("bootstrap_grouped_uid_ci")


def plot_error_counts():
    if error_counts is None:
        print("No hay error counts.")
        return
    df = error_counts.copy()
    df = df[df["model"].isin(["image_calibrated", "fusion_weighted", "fusion_stack", "text_calibrated"])]
    x = np.arange(len(df))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - 0.2, df["FP"], width=0.4, label="FP")
    ax.bar(x + 0.2, df["FN"], width=0.4, label="FN")
    ax.set_xticks(x)
    ax.set_xticklabels(df["model"], rotation=20, ha="right")
    ax.set_ylabel("Casos")
    ax.set_title("Errores al umbral primario")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    save_current_fig("error_counts_primary_threshold")

plot_cv_aggregate()
plot_bootstrap_ci()
plot_error_counts()

if mcnemar is not None:
    mcnemar.to_csv(TABLE_DIR / f"mcnemar_{VERSION}.csv", index=False)
    print("McNemar:")
    display(mcnemar)

## 9. Interpretación automática para informe o presentación

Esta celda genera un resumen en Markdown con los puntos clave y lo guarda en `docs/presentation_summary_v6_1.md`.

In [ ]:
# =============================
# 12) Resumen interpretativo
# =============================
def fmt_pct(x):
    if pd.isna(x):
        return "NA"
    return f"{100*x:.1f}%"

def fmt3(x):
    if pd.isna(x):
        return "NA"
    return f"{x:.3f}"

primary_row = main_uid_primary[main_uid_primary["model"].eq(PRIMARY_MODEL)].iloc[0]
weighted_row = main_uid_primary[main_uid_primary["model"].eq("fusion_weighted")].iloc[0] if (main_uid_primary["model"].eq("fusion_weighted")).any() else None
stack_row = main_uid_primary[main_uid_primary["model"].eq("fusion_stack")].iloc[0] if (main_uid_primary["model"].eq("fusion_stack")).any() else None
text_row = main_uid_primary[main_uid_primary["model"].eq("text_calibrated")].iloc[0] if (main_uid_primary["model"].eq("text_calibrated")).any() else None

fusion_weight_text = "No disponible"
if fusion_selected:
    iw = fusion_selected.get("image_weight", fusion_selected.get("best_image_weight", None))
    tw = fusion_selected.get("text_weight", fusion_selected.get("best_text_weight", None))
    if iw is not None and tw is not None:
        fusion_weight_text = f"image_weight={iw}, text_weight={tw}"

summary_md = f"""
# Resumen ejecutivo V6.1

## Objetivo
Desarrollar y evaluar un sistema para clasificar radiografías de tórax de IU X-Ray como **normal** vs **anormalidad torácica no incidental**, evitando fuga textual desde `findings` o `impression`.

## Pipeline

### Data input
- Dataset: Indiana University Chest X-ray Collection.
- Entrada visual: radiografía frontal.
- Entrada textual segura: `indication` clínica.
- Unidad de separación: UID del estudio, para evitar fuga entre imágenes del mismo caso.

### Preprocessing
- Limpieza y normalización de texto.
- Auditoría de etiquetas con reglas para negaciones, `no indexing`, hallazgos hipotéticos, hallazgos resueltos y nódulos/granulomas benignos.
- Exclusión de casos ambiguos o incidentales.
- División en `train_fit`, `model_val`, `calibration`, `threshold` y `test`.

### Model
- Rama visual entrenada con imágenes frontales.
- Rama textual basada solo en `indication`.
- Calibración de probabilidades.
- Fusión evaluada mediante `fusion_weighted` y `fusion_stack`.

### Training
- Entrenamiento principal con splits separados.
- Validación cruzada por UID.
- Selección de umbral en split independiente.

### Evaluation
- Métricas principales: ROC-AUC, PR-AUC, Brier, sensibilidad, especificidad, FPR, F1.
- Métricas adicionales: NPV, balanced accuracy, MCC, Youden J, LR+ y LR-.
- Bootstrap agrupado por UID.
- McNemar para comparar errores.

### Final output
- Modelo recomendado: `{PRIMARY_MODEL}`.
- Umbral operativo: `{PRIMARY_THRESHOLD_TYPE}`.
- Paquete reproducible para GitHub y demo.

## Resultados principales por UID

| Métrica | {PRIMARY_MODEL} |
|---|---:|
| ROC-AUC | {fmt3(primary_row['roc_auc'])} |
| PR-AUC | {fmt3(primary_row['pr_auc'])} |
| Brier | {fmt3(primary_row['brier'])} |
| Sensibilidad | {fmt_pct(primary_row['recall_sensitivity'])} |
| Especificidad | {fmt_pct(primary_row['specificity'])} |
| FPR | {fmt_pct(primary_row['false_positive_rate'])} |
| F1 | {fmt3(primary_row['f1'])} |
| TN / FP / FN / TP | {int(primary_row['tn'])} / {int(primary_row['fp'])} / {int(primary_row['fn'])} / {int(primary_row['tp'])} |

## Hallazgo sobre multimodalidad
- Peso seleccionado para fusión ponderada: `{fusion_weight_text}`.
- La señal textual basada solo en `indication` no aportó mejora robusta frente al modelo visual calibrado.
- La hipótesis multimodal queda parcialmente rechazada bajo esta configuración: el texto seguro evita fuga, pero su capacidad predictiva fue limitada.

## Conclusión recomendada
V6.1 representa la versión más limpia del target. El modelo visual calibrado debe presentarse como resultado principal. La fusión puede reportarse como análisis exploratorio, explicando que no redujo falsos positivos de forma robusta y que el selector conservador prefirió imagen pura.
""".strip()

summary_path = DOCS_DIR / f"presentation_summary_{VERSION}.md"
summary_path.write_text(summary_md, encoding="utf-8")
print(summary_path)
display(Markdown(summary_md))

## 10. Demo recomendada para la exposición

Para una presentación, lo más estable es una **demo de inferencia/evaluación retrospectiva**, no reentrenar en vivo. La demo debe mostrar:

1. Una imagen/caso de test.
2. La `indication` clínica.
3. La probabilidad del modelo.
4. El umbral operativo.
5. La decisión final.
6. Si fue OK, FP o FN.

Esto demuestra funcionalidad sin depender de GPU ni de tiempos de entrenamiento.

In [ ]:
# =============================
# 13) Demo retrospectiva en notebook
# =============================
def get_threshold_for_model(model=PRIMARY_MODEL, threshold_type=PRIMARY_THRESHOLD_TYPE):
    row = thresholds_main[(thresholds_main["model"].eq(model)) & (thresholds_main["threshold_type"].eq(threshold_type))]
    if row.empty:
        # fallback desde summary_main
        row = summary_main[(summary_main["model"].eq(model)) & (summary_main["threshold_type"].eq(threshold_type))]
    if row.empty:
        return None
    return float(row.iloc[0]["threshold"])

def prob_col_for_model(model):
    mapping = {
        "image_raw": "prob_image_raw",
        "image_calibrated": "prob_image_cal",
        "text_calibrated": "prob_text_cal",
        "fusion_weighted": "prob_fusion_weighted",
        "fusion_stack": "prob_fusion_stack",
    }
    return mapping.get(model)


def select_demo_row(kind="true_positive", model=PRIMARY_MODEL):
    if error_analysis is None:
        raise ValueError("No hay error_analysis.")
    err_col = f"error_{model}"
    pred_col = f"pred_{model}"
    if err_col not in error_analysis.columns:
        # Algunos nombres tienen calibrado completo.
        if model == "image_calibrated":
            err_col = "error_image_calibrated"
            pred_col = "pred_image_calibrated"
    df = error_analysis.copy()
    if kind == "true_positive":
        cand = df[(df["label"].eq(1)) & (df[err_col].eq("OK"))]
    elif kind == "true_negative":
        cand = df[(df["label"].eq(0)) & (df[err_col].eq("OK"))]
    elif kind == "false_positive":
        cand = df[df[err_col].eq("FP")]
    elif kind == "false_negative":
        cand = df[df[err_col].eq("FN")]
    else:
        cand = df.sample(min(len(df), 20), random_state=7)
    if cand.empty:
        cand = df.sample(min(len(df), 20), random_state=7)
    return cand.sample(1, random_state=7).iloc[0]


def show_demo_case(kind="true_positive", model=PRIMARY_MODEL):
    row = select_demo_row(kind=kind, model=model)
    threshold = get_threshold_for_model(model=model)
    pcol = prob_col_for_model(model)
    prob = float(row[pcol]) if pcol in row else np.nan
    pred = int(prob >= threshold) if threshold is not None and not np.isnan(prob) else None
    label = int(row["label"])
    decision = "ANORMALIDAD NO INCIDENTAL" if pred == 1 else "NORMAL / NO RELEVANTE"
    truth = "POSITIVO" if label == 1 else "NEGATIVO"
    error_col = f"error_{model}"
    if error_col not in row and model == "image_calibrated":
        error_col = "error_image_calibrated"
    status = row.get(error_col, "NA")

    md_text = f"""
### Demo de caso — `{kind}`

| Campo | Valor |
|---|---|
| UID | `{row.get('uid', 'NA')}` |
| Archivo | `{row.get('filename', 'NA')}` |
| Categoría diagnóstica | `{row.get('diagnostic_category', 'NA')}` |
| Label real | `{truth}` |
| Modelo | `{model}` |
| Probabilidad | `{prob:.3f}` |
| Umbral | `{threshold:.3f}` |
| Decisión | **{decision}** |
| Resultado | `{status}` |

**Indication:** {row.get('indication', 'NA')}

**Findings/Impression, solo para explicación retrospectiva y NO como entrada del modelo:**  
{str(row.get('findings', ''))[:500]}  
{str(row.get('impression', ''))[:300]}
"""
    display(Markdown(md_text))

    # Mostrar imagen si existe localmente.
    img_path = row.get("image_path", None)
    if isinstance(img_path, str) and Path(img_path).exists():
        try:
            from PIL import Image
            img = Image.open(img_path)
            display(img.resize((min(512, img.width), int(img.height * min(512, img.width) / img.width))))
        except Exception as e:
            print("No pude mostrar imagen:", e)
    else:
        print("Imagen no disponible localmente. Ruta esperada:", img_path)
        print("Para mostrarla, monta/carga el dataset IU X-Ray en la ruta indicada o adapta image_path.")

# Ejecuta una demo positiva correcta.
show_demo_case(kind="true_positive", model=PRIMARY_MODEL)

### Opcional: demo con Gradio

Activa `INSTALL_GRADIO=True` si quieres abrir una interfaz interactiva. Para exposición, puedes ejecutar esta celda y seleccionar casos `true_positive`, `true_negative`, `false_positive` o `false_negative`.

In [ ]:
# =============================
# 14) Demo opcional con Gradio
# =============================
INSTALL_GRADIO = False

if INSTALL_GRADIO:
    try:
        import gradio as gr
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
        import gradio as gr

    def demo_gradio(kind, model):
        row = select_demo_row(kind=kind, model=model)
        threshold = get_threshold_for_model(model=model)
        pcol = prob_col_for_model(model)
        prob = float(row[pcol]) if pcol in row else np.nan
        pred = int(prob >= threshold) if threshold is not None and not np.isnan(prob) else None
        decision = "ANORMALIDAD NO INCIDENTAL" if pred == 1 else "NORMAL / NO RELEVANTE"
        label = int(row["label"])
        truth = "POSITIVO" if label == 1 else "NEGATIVO"
        err_col = f"error_{model}"
        if err_col not in row and model == "image_calibrated":
            err_col = "error_image_calibrated"
        status = row.get(err_col, "NA")
        report = f"""UID: {row.get('uid', 'NA')}\nArchivo: {row.get('filename', 'NA')}\nCategoría: {row.get('diagnostic_category', 'NA')}\nLabel real: {truth}\nModelo: {model}\nProbabilidad: {prob:.3f}\nUmbral: {threshold:.3f}\nDecisión: {decision}\nResultado: {status}\n\nIndication: {row.get('indication', 'NA')}"""
        img_path = row.get("image_path", None)
        if isinstance(img_path, str) and Path(img_path).exists():
            return img_path, report
        return None, report + "\n\nImagen no disponible localmente."

    iface = gr.Interface(
        fn=demo_gradio,
        inputs=[
            gr.Dropdown(["true_positive", "true_negative", "false_positive", "false_negative", "random"], value="true_positive", label="Tipo de caso"),
            gr.Dropdown(["image_calibrated", "fusion_weighted", "fusion_stack", "text_calibrated"], value=PRIMARY_MODEL, label="Modelo"),
        ],
        outputs=[gr.Image(label="Radiografía"), gr.Textbox(label="Resultado", lines=12)],
        title="Demo diagnóstico IU X-Ray V6.1",
        description="Demo retrospectiva usando predicciones del conjunto test. No reentrena el modelo."
    )
    iface.launch(share=True)
else:
    print("Gradio desactivado. Cambia INSTALL_GRADIO=True para usar interfaz interactiva.")

## 11. Crear paquete para GitHub

El repositorio generado cumple con lo solicitado:

- Source code
- README file
- Dataset link or instructions
- Execution steps

**Nota:** este notebook no puede crear automáticamente un repositorio remoto en tu cuenta sin credenciales. Sí deja la carpeta lista para que la subas a GitHub y obtengas el enlace.

In [ ]:
# =============================
# 15) Crear estructura GitHub-ready
# =============================
def write_text(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")

# Limpiar repo generado.
if GITHUB_DIR.exists():
    shutil.rmtree(GITHUB_DIR)
GITHUB_DIR.mkdir(parents=True, exist_ok=True)

# Directorios.
for sub in ["src", "notebooks", "data", "results/tables", "assets/figures", "docs", "demo"]:
    (GITHUB_DIR / sub).mkdir(parents=True, exist_ok=True)

# Copiar figuras y tablas generadas.
for p in FIG_DIR.glob("*"):
    if p.is_file():
        shutil.copy2(p, GITHUB_DIR / "assets" / "figures" / p.name)
for p in TABLE_DIR.glob("*.csv"):
    shutil.copy2(p, GITHUB_DIR / "results" / "tables" / p.name)
for p in DOCS_DIR.glob("*.md"):
    shutil.copy2(p, GITHUB_DIR / "docs" / p.name)

# Copiar ZIP de outputs como referencia opcional? Por tamaño, no se copia. Mejor instrucciones.

readme = f"""
# Diagnóstico multimodal en radiografías de tórax IU X-Ray — V6.1

Este repositorio contiene el código, resultados y materiales de reporte del proyecto de clasificación binaria de radiografías de tórax de Indiana University X-Ray.

## Objetivo

Clasificar estudios como:

- `0`: normal / no relevante para la tarea binaria.
- `1`: anormalidad torácica no incidental.

La versión V6.1 usa una definición conservadora del target con auditoría de negaciones, hallazgos hipotéticos, hallazgos resueltos y nódulos/granulomas benignos.

## Pipeline

1. **Data input:** radiografías frontales, `indication` clínica y metadata por UID.
2. **Preprocessing:** normalización, filtrado frontal, auditoría de etiquetas, exclusión de casos ambiguos/incidental/resueltos.
3. **Model:** modelo visual calibrado y modelos exploratorios de texto/fusión.
4. **Training:** splits separados por UID: train, validation, calibration, threshold y test.
5. **Evaluation:** ROC-AUC, PR-AUC, Brier, sensibilidad, especificidad, FPR, métricas adicionales, bootstrap por UID y McNemar.
6. **Final output:** modelo recomendado `image_calibrated` y materiales de reporte.

## Resultado principal

Modelo recomendado: `{PRIMARY_MODEL}`  
Umbral operativo: `{PRIMARY_THRESHOLD_TYPE}`

| Métrica UID | Valor |
|---|---:|
| ROC-AUC | {fmt3(primary_row['roc_auc'])} |
| PR-AUC | {fmt3(primary_row['pr_auc'])} |
| Brier | {fmt3(primary_row['brier'])} |
| Sensibilidad | {fmt_pct(primary_row['recall_sensitivity'])} |
| Especificidad | {fmt_pct(primary_row['specificity'])} |
| FPR | {fmt_pct(primary_row['false_positive_rate'])} |
| F1 | {fmt3(primary_row['f1'])} |

## Dataset

Dataset utilizado: **Indiana University Chest X-ray Collection**.

Instrucciones generales:

1. Crear una cuenta de Kaggle.
2. Configurar credenciales de Kaggle o usar `kagglehub` en Colab.
3. Descargar el dataset de Indiana University Chest X-ray Collection.
4. Mantener la estructura de imágenes y reportes esperada por el notebook de entrenamiento.

Por restricciones de tamaño/licencia, las imágenes no se versionan en este repositorio.

## Ejecución

### 1. Entrenamiento

Ejecutar el notebook principal de entrenamiento en Colab con GPU:

```bash
notebooks/diagnostico_multimodal_iu_xray_v6_1_colab.ipynb
```

El entrenamiento genera:

```text
outputs_multimodal_iuxray_v6_1.zip
```

### 2. Reporte y assets

Ejecutar el notebook de reporting:

```bash
notebooks/diagnostico_multimodal_iu_xray_v6_1_reporting_github_demo_colab.ipynb
```

Este genera figuras, tablas, documentación y demo.

### 3. Demo

La demo recomendada es retrospectiva: toma un caso del conjunto test, muestra la radiografía si está disponible, la indicación clínica, la probabilidad del modelo, el umbral y la decisión final.

```bash
python demo/app_demo_gradio.py
```

## Estructura

```text
├── src/
├── notebooks/
├── data/
├── results/
├── assets/figures/
├── docs/
└── demo/
```

## Conclusión

V6.1 es la versión más limpia del target. El modelo visual calibrado se recomienda como modelo principal. La fusión multimodal se reporta como análisis exploratorio, ya que el texto seguro basado solo en `indication` no aportó una mejora robusta.
"""
write_text(GITHUB_DIR / "README.md", readme)

write_text(GITHUB_DIR / "requirements.txt", """
numpy
pandas
matplotlib
scikit-learn
Pillow
jupyter
ipywidgets
gradio
""")

write_text(GITHUB_DIR / ".gitignore", """
__pycache__/
*.pyc
.ipynb_checkpoints/
.env
.DS_Store
outputs_multimodal_iuxray_*.zip
*.pt
*.pth
/content/
data/raw/
data/images/
""")

write_text(GITHUB_DIR / "data" / "README_dataset.md", """
# Dataset instructions

Este proyecto utiliza Indiana University Chest X-ray Collection.

No se incluyen imágenes ni reportes crudos en el repositorio.

## Pasos sugeridos

1. Crear una cuenta en Kaggle.
2. Obtener credenciales de Kaggle si se ejecutará desde Colab.
3. Descargar el dataset IU X-Ray con KaggleHub o desde Kaggle.
4. Mantener las rutas esperadas por el notebook de entrenamiento.
5. Ejecutar el notebook principal para reconstruir `outputs_multimodal_iuxray_v6_1.zip`.

El proyecto usa únicamente `indication` como texto de entrada para evitar fuga de información desde `findings` o `impression`.
""")

write_text(GITHUB_DIR / "docs" / "pipeline.md", """
# Pipeline del proyecto

## Data input
- Radiografías frontales de tórax.
- `indication` clínica como texto seguro.
- UID del estudio para separación sin fuga.

## Preprocessing
- Filtrado por proyección frontal.
- Normalización de imagen.
- Normalización de texto.
- Auditoría de etiquetas.
- Exclusión de casos ambiguos, incidentales, resueltos o benignos nodulares.

## Model
- Rama visual calibrada.
- Rama textual exploratoria.
- Fusión ponderada y stacking como modelos secundarios.

## Training
- Separación por UID.
- Splits: train_fit, model_val, calibration, threshold y test.
- Validación cruzada por UID.

## Evaluation
- ROC-AUC, PR-AUC, Brier.
- Sensibilidad, especificidad, FPR, FNR.
- F1, NPV, balanced accuracy, MCC, Youden J.
- Bootstrap agrupado por UID.
- McNemar.

## Final output
- Modelo principal: image_calibrated.
- ZIP de outputs.
- Assets de presentación.
- Paquete reproducible para GitHub.
""")

write_text(GITHUB_DIR / "docs" / "demo_script.md", f"""
# Guion sugerido para la demo en vivo

Duración sugerida: 2–4 minutos.

## Paso 1: Contexto

Explicar que el entrenamiento completo ya fue ejecutado previamente en GPU y que la demo muestra inferencia/evaluación retrospectiva sobre casos del test.

## Paso 2: Entrada

Mostrar un caso con:

- Radiografía frontal.
- Indication clínica.
- UID del estudio.

## Paso 3: Modelo

Indicar que el modelo principal es `{PRIMARY_MODEL}`.

## Paso 4: Salida

Mostrar:

- Probabilidad estimada.
- Umbral operativo `{PRIMARY_THRESHOLD_TYPE}`.
- Decisión final.
- Comparación con label real.

## Paso 5: Interpretación

Frase sugerida:

> Esta demo no reentrena el modelo en vivo; ejecuta la etapa de inferencia/evaluación sobre casos reservados del test, que es la forma más estable de demostrar funcionalidad en una exposición.

## Casos recomendados

1. True positive: muestra detección correcta.
2. True negative: muestra control de falsos positivos.
3. False positive: permite discutir limitaciones.
4. False negative: permite discutir seguridad y sensibilidad.
""")

write_text(GITHUB_DIR / "src" / "metrics.py", r"""
import math
import numpy as np


def safe_div(a, b):
    return np.nan if b == 0 else a / b


def binary_metrics_from_confusion(tn, fp, fn, tp):
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    fpr = safe_div(fp, fp + tn)
    fnr = safe_div(fn, fn + tp)
    precision = safe_div(tp, tp + fp)
    npv = safe_div(tn, tn + fn)
    accuracy = safe_div(tp + tn, tp + tn + fp + fn)
    f1 = safe_div(2 * tp, 2 * tp + fp + fn)
    balanced_accuracy = np.nanmean([sensitivity, specificity])
    denom = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    mcc = safe_div((tp*tn - fp*fn), denom)
    youden_j = sensitivity + specificity - 1
    lr_positive = safe_div(sensitivity, fpr)
    lr_negative = safe_div(fnr, specificity)
    return {
        "accuracy": accuracy,
        "precision_ppv": precision,
        "npv": npv,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "fpr": fpr,
        "fnr": fnr,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "mcc": mcc,
        "youden_j": youden_j,
        "lr_positive": lr_positive,
        "lr_negative": lr_negative,
    }
""")

write_text(GITHUB_DIR / "src" / "make_report.py", r"""
from pathlib import Path
import pandas as pd


def load_final_metrics(results_dir="results/tables"):
    results_dir = Path(results_dir)
    files = sorted(results_dir.glob("final_metrics_test_uid_*.csv"))
    if not files:
        raise FileNotFoundError("No final_metrics_test_uid_*.csv found")
    return pd.read_csv(files[-1])


if __name__ == "__main__":
    df = load_final_metrics()
    print(df[["model", "roc_auc", "pr_auc", "brier", "recall_sensitivity", "specificity", "false_positive_rate"]])
""")

# Guardar predicciones para demo si están disponibles.
if error_analysis is not None:
    error_analysis.to_csv(GITHUB_DIR / "demo" / f"error_analysis_test_{VERSION}.csv", index=False)
if thresholds_main is not None:
    thresholds_main.to_csv(GITHUB_DIR / "demo" / f"thresholds_main_{VERSION}.csv", index=False)

write_text(GITHUB_DIR / "demo" / "app_demo_gradio.py", f"""
from pathlib import Path
import pandas as pd

try:
    import gradio as gr
except ImportError as exc:
    raise SystemExit("Instala gradio con: pip install gradio") from exc

VERSION = "{VERSION}"
PRIMARY_MODEL = "{PRIMARY_MODEL}"
PRIMARY_THRESHOLD_TYPE = "{PRIMARY_THRESHOLD_TYPE}"
BASE = Path(__file__).resolve().parent
ERROR_CSV = BASE / f"error_analysis_test_{{VERSION}}.csv"
THRESHOLD_CSV = BASE / f"thresholds_main_{{VERSION}}.csv"

error_df = pd.read_csv(ERROR_CSV)
thr_df = pd.read_csv(THRESHOLD_CSV)


def threshold_for(model):
    row = thr_df[(thr_df["model"] == model) & (thr_df["threshold_type"] == PRIMARY_THRESHOLD_TYPE)]
    if row.empty:
        return float(thr_df[thr_df["model"] == model].iloc[0]["threshold"])
    return float(row.iloc[0]["threshold"])


def prob_col(model):
    return {{
        "image_raw": "prob_image_raw",
        "image_calibrated": "prob_image_cal",
        "text_calibrated": "prob_text_cal",
        "fusion_weighted": "prob_fusion_weighted",
        "fusion_stack": "prob_fusion_stack",
    }}[model]


def error_col(model):
    if model == "image_calibrated":
        return "error_image_calibrated"
    return f"error_{{model}}"


def select_case(kind, model):
    e_col = error_col(model)
    df = error_df.copy()
    if kind == "true_positive":
        cand = df[(df["label"] == 1) & (df[e_col] == "OK")]
    elif kind == "true_negative":
        cand = df[(df["label"] == 0) & (df[e_col] == "OK")]
    elif kind == "false_positive":
        cand = df[df[e_col] == "FP"]
    elif kind == "false_negative":
        cand = df[df[e_col] == "FN"]
    else:
        cand = df
    if cand.empty:
        cand = df
    return cand.sample(1, random_state=7).iloc[0]


def demo(kind, model):
    row = select_case(kind, model)
    thr = threshold_for(model)
    p = float(row[prob_col(model)])
    pred = int(p >= thr)
    decision = "ANORMALIDAD NO INCIDENTAL" if pred else "NORMAL / NO RELEVANTE"
    truth = "POSITIVO" if int(row["label"]) else "NEGATIVO"
    status = row.get(error_col(model), "NA")
    text = (
        f"UID: {{row.get('uid', 'NA')}}\n"
        f"Archivo: {{row.get('filename', 'NA')}}\n"
        f"Categoría: {{row.get('diagnostic_category', 'NA')}}\n"
        f"Label real: {{truth}}\n"
        f"Modelo: {{model}}\n"
        f"Probabilidad: {{p:.3f}}\n"
        f"Umbral: {{thr:.3f}}\n"
        f"Decisión: {{decision}}\n"
        f"Resultado: {{status}}\n\n"
        f"Indication: {{row.get('indication', 'NA')}}"
    )
    img_path = row.get("image_path", None)
    if isinstance(img_path, str) and Path(img_path).exists():
        return img_path, text
    return None, text + "\n\nImagen no disponible localmente."


iface = gr.Interface(
    fn=demo,
    inputs=[
        gr.Dropdown(["true_positive", "true_negative", "false_positive", "false_negative", "random"], value="true_positive", label="Tipo de caso"),
        gr.Dropdown(["image_calibrated", "fusion_weighted", "fusion_stack", "text_calibrated"], value=PRIMARY_MODEL, label="Modelo"),
    ],
    outputs=[gr.Image(label="Radiografía"), gr.Textbox(label="Resultado", lines=12)],
    title="Demo diagnóstico IU X-Ray V6.1",
    description="Demo retrospectiva usando predicciones del conjunto test. No reentrena el modelo."
)

if __name__ == "__main__":
    iface.launch()
""")

# README en notebooks.
write_text(GITHUB_DIR / "notebooks" / "README.md", """
# Notebooks

Coloca aquí los notebooks:

- `diagnostico_multimodal_iu_xray_v6_1_colab.ipynb`: entrenamiento principal.
- `diagnostico_multimodal_iu_xray_v6_1_reporting_github_demo_colab.ipynb`: reporte, figuras, GitHub package y demo.

Si los notebooks son grandes, también puedes enlazarlos desde Google Colab.
""")

# Intentar copiar notebooks si existen localmente.
candidate_notebooks = [
    Path("/content/diagnostico_multimodal_iu_xray_v6_1_colab.ipynb"),
    Path("/content/diagnostico_multimodal_iu_xray_v6_1_reporting_github_demo_colab.ipynb"),
    Path("/mnt/data/diagnostico_multimodal_iu_xray_v6_1_colab.ipynb"),
    Path("/mnt/data/diagnostico_multimodal_iu_xray_v6_1_reporting_github_demo_colab.ipynb"),
]
for nbp in candidate_notebooks:
    if nbp.exists():
        shutil.copy2(nbp, GITHUB_DIR / "notebooks" / nbp.name)

print("Repositorio generado en:", GITHUB_DIR)
for p in sorted(GITHUB_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(GITHUB_DIR))

## 12. Cómo obtener el GitHub Link

Después de generar la carpeta `github_repo`, tienes dos opciones:

### Opción A: subir desde la web de GitHub

1. Entra a GitHub.
2. Crea un repositorio nuevo.
3. Sube los archivos de la carpeta `github_repo`.
4. Copia el enlace del repositorio y úsalo como **GitHub Link**.

### Opción B: subir con Git desde Colab/local

Crea un repositorio vacío en GitHub y pega su URL en `GITHUB_REPO_URL`. Luego ejecuta la celda.

In [ ]:
# =============================
# 16) Opcional: preparar push a GitHub
# =============================
# Ejemplo:
# GITHUB_REPO_URL = "https://github.com/tu_usuario/diagnostico-multimodal-iuxray.git"
GITHUB_REPO_URL = ""

if GITHUB_REPO_URL:
    old_cwd = os.getcwd()
    os.chdir(GITHUB_DIR)
    subprocess.run(["git", "init"], check=False)
    subprocess.run(["git", "add", "."], check=False)
    subprocess.run(["git", "commit", "-m", "Initial release V6.1"], check=False)
    subprocess.run(["git", "branch", "-M", "main"], check=False)
    subprocess.run(["git", "remote", "remove", "origin"], check=False)
    subprocess.run(["git", "remote", "add", "origin", GITHUB_REPO_URL], check=False)
    print("Ahora ejecuta manualmente si tienes autenticación configurada:")
    print("git push -u origin main")
    os.chdir(old_cwd)
else:
    print("GITHUB_REPO_URL está vacío. Crea el repo en GitHub y pega aquí la URL si quieres hacer push desde el entorno.")
    print("Carpeta lista para subir:", GITHUB_DIR)

## 13. Crear ZIPs finales

Se generarán dos ZIPs:

1. `presentation_assets_v6_1.zip`: figuras, tablas y docs para la presentación.
2. `multimodal_iuxray_v6_1_github_repo_ready.zip`: repositorio listo para subir a GitHub.

In [ ]:
# =============================
# 17) Comprimir entregables
# =============================
def make_zip(src_dir, zip_path):
    src_dir = Path(src_dir)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(zip_path.with_suffix("").as_posix(), "zip", src_dir)
    return zip_path

# Carpeta de assets de presentación.
PRES_DIR = REPORT_DIR / "presentation_assets"
if PRES_DIR.exists():
    shutil.rmtree(PRES_DIR)
PRES_DIR.mkdir(parents=True, exist_ok=True)
for name, src in [("figures", FIG_DIR), ("tables", TABLE_DIR), ("docs", DOCS_DIR)]:
    dst = PRES_DIR / name
    shutil.copytree(src, dst)

presentation_zip = make_zip(PRES_DIR, REPORT_DIR / f"presentation_assets_{VERSION}.zip")
github_zip = make_zip(GITHUB_DIR, REPORT_DIR / f"multimodal_iuxray_{VERSION}_github_repo_ready.zip")

print("ZIP presentación:", presentation_zip)
print("ZIP GitHub-ready:", github_zip)
print("\nArchivos finales en REPORT_DIR:")
for p in sorted(REPORT_DIR.glob("*.zip")):
    print(" -", p.name, f"({p.stat().st_size/1024/1024:.2f} MB)")

## 14. Checklist final para la presentación

Usa esta lista para verificar que tienes todo:

- [ ] Figura `pipeline_project_flow.png` para explicar Data input → Final output.
- [ ] Figura `model_architecture_v6_1.png` para explicar el modelo.
- [ ] Gráficos de distribución de clases y splits.
- [ ] Tabla de métricas principales por UID.
- [ ] Gráficos ROC/PR/calibración.
- [ ] Gráficos de CV, FPR y errores.
- [ ] Guion de demo en `docs/demo_script.md`.
- [ ] Carpeta `github_repo` o ZIP `multimodal_iuxray_v6_1_github_repo_ready.zip`.
- [ ] Repositorio subido a GitHub con README, código fuente, dataset instructions y execution steps.

### Recomendación para la demo

Haz una demo retrospectiva con un caso positivo correcto y luego un falso positivo. Esto demuestra funcionalidad y también muestra pensamiento crítico sobre limitaciones del sistema.